# eCooking mini-grid model — v5 Colab driver

All code now lives in the repository; this notebook only runs it.
v4's `%%writefile` cells are retired, so the notebook and the paper's code
cannot drift apart.

1. Install (then **Runtime → Restart session**)
2. Data: NASA POWER is fetched; Kenya MTF is fetched; Rwanda MTF is uploaded (licensed)
3. `pipeline.py all` regenerates every result, table and figure
4. Tests, and the manuscript-vs-results check

In [ ]:
!git clone -q https://github.com/chilalaabubakar/EDA.git ecook 2>/dev/null || (cd ecook && git pull -q)
%cd ecook
!pip install -q -r requirements.txt 2>&1 | tail -1
print("NOW: Runtime -> Restart session, then run from the next cell")

In [ ]:
%cd /content/ecook
import numpy; assert numpy.__version__.startswith("1."), "restart the session first"
!python src/pipeline.py resource          # NASA POWER 2005-2024, both sites, LST

## Survey microdata (needed only to re-derive the MTF inputs)

The derived appliance sets are committed in `src/final_cfg.py`, so the model
runs without this step. Rwanda MTF 2022 is licensed: download
`RWA_2022_MTF_v01_M_CSV.zip` from catalog 6429 and put it in Drive `/mtf/`
(see `docs/DATA_ACCESS.md`). Never commit it.

In [ ]:
USE_DRIVE = True
import glob, zipfile, os
os.makedirs("data/raw", exist_ok=True)
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    for z in glob.glob("/content/drive/MyDrive/mtf/*.zip"):
        zipfile.ZipFile(z).extractall("data/raw"); print("extracted", os.path.basename(z))
if os.path.exists("data/raw/household_survey_data/SECTION_EF1.csv"):
    !python src/pipeline.py mtf
else:
    print("microdata absent - using the committed MTF-derived values")

In [ ]:
# Phases 1-4: sweep, bands, validation, viability, tables, expansion, sizing
# check, interannual, 24-seed ensemble, ToU sweep, ESMAP, operating subsidy,
# household bands, peaks, Ringiti, household cost, figures.
!python src/pipeline.py all

In [ ]:
!python -m pytest -q
!cat results/tables/tables.md
!python src/manuscript_tables.py --abstract

## Manuscript check

Upload the current draft; every table cell and headline number is compared
with `results/`. Mismatches list exactly what to change.

In [ ]:
from google.colab import files
up = files.upload()
os.makedirs("manuscript/private", exist_ok=True)
for name, data in up.items():
    open(f"manuscript/private/{name}", "wb").write(data)
    !python src/manuscript_check.py "manuscript/private/{name}"
    !python src/word_count.py "manuscript/private/{name}" --limit 5000

In [ ]:
# Commit results (not data, not the manuscript) back to the repo, if desired:
# !git add results data/processed && git commit -m "Results from Colab run" && git push
!zip -qr results.zip results && echo results.zip ready
files.download("results.zip")